# 04 — Movie Industry Analysis

This notebook turns the EDA into business-style movie industry insights.

Focus:
- yearly movie volume and score
- genre composition
- financial performance
- profitability and ROI
- budget efficiency
- release timing
- production companies and directors

All conclusions should be described as patterns in **this dataset**, not as universal industry facts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

movies = pd.read_csv("../data/processed/movies_cleaned.csv")
movies = movies.dropna(how="all").reset_index(drop=True)

for col in ["year", "score", "votes", "budget", "gross", "runtime", "release_year"]:
    movies[col] = pd.to_numeric(movies[col], errors="coerce")

movies["released_date"] = pd.to_datetime(movies["released_date"], errors="coerce")

print("Shape:", movies.shape)

## 1. Yearly overview

In [ ]:
year_summary = movies.groupby("year").agg(
    movie_count=("name", "count"),
    average_score=("score", "mean"),
    median_score=("score", "median"),
    average_runtime=("runtime", "mean"),
    average_votes=("votes", "mean")
).reset_index()

year_summary

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

sns.lineplot(data=year_summary, x="year", y="movie_count", marker="o", ax=ax1)
ax1.set_title("Movie Count by Year")
ax1.set_xlabel("Year")
ax1.set_ylabel("Movie Count")

plt.tight_layout()
plt.show()

## 2. Genre profile

In [ ]:
genre_summary = movies.groupby("genre").agg(
    movie_count=("name", "count"),
    average_score=("score", "mean"),
    median_score=("score", "median"),
    average_runtime=("runtime", "mean"),
    average_votes=("votes", "mean")
).sort_values("movie_count", ascending=False)

genre_summary

In [ ]:
genre_summary["share_percent"] = (
    genre_summary["movie_count"] / len(movies) * 100
).round(2)

genre_summary

## 3. Financial performance by genre

In [ ]:
financial = movies[movies["budget"].notna() & movies["gross"].notna() & (movies["budget"] > 0)].copy()
financial["profit"] = financial["gross"] - financial["budget"]
financial["roi_percent"] = (financial["profit"] / financial["budget"]) * 100

genre_financial = financial.groupby("genre").agg(
    movies_with_financial_data=("name", "count"),
    median_budget=("budget", "median"),
    median_gross=("gross", "median"),
    median_profit=("profit", "median"),
    median_roi_percent=("roi_percent", "median")
).sort_values("median_profit", ascending=False)

genre_financial

In [ ]:
plt.figure(figsize=(12, 7))
plot_data = genre_financial[genre_financial["movies_with_financial_data"] >= 20]

sns.barplot(
    data=plot_data.reset_index(),
    x="median_profit",
    y="genre"
)

plt.title("Median Profit by Genre (At Least 20 Financial Records)")
plt.xlabel("Median Profit")
plt.ylabel("Genre")
plt.tight_layout()
plt.show()

## 4. Highest-grossing and highest-profit movies

In [ ]:
top_gross = movies[movies["gross"].notna()][
    ["name", "gross", "budget", "score", "genre", "year"]
].sort_values("gross", ascending=False)

top_gross.head(20)

In [ ]:
top_profit = financial[
    ["name", "budget", "gross", "profit", "roi_percent", "score", "genre", "year"]
].sort_values("profit", ascending=False)

top_profit.head(20)

## 5. Budget efficiency

In [ ]:
roi_table = financial[
    ["name", "budget", "gross", "profit", "roi_percent", "score", "genre", "year"]
].copy()

roi_table[roi_table["budget"] >= 1_000_000].sort_values(
    "roi_percent", ascending=False
).head(20)

## 6. Score and financial performance

In [ ]:
print("Budget vs gross:", round(financial["budget"].corr(financial["gross"]), 4))
print("Score vs gross:", round(financial["score"].corr(financial["gross"]), 4))
print("Score vs profit:", round(financial["score"].corr(financial["profit"]), 4))
print("Score vs ROI:", round(financial["score"].corr(financial["roi_percent"]), 4))

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(data=financial, x="score", y="gross", alpha=0.45)
plt.yscale("log")
plt.title("Movie Score vs Gross Revenue")
plt.xlabel("Score")
plt.ylabel("Gross Revenue (log scale)")
plt.tight_layout()
plt.show()

## 7. Release month analysis

In [ ]:
month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]

month_summary = movies.groupby("release_month").agg(
    movie_count=("name", "count"),
    average_score=("score", "mean"),
    median_score=("score", "median")
).reindex(month_order)

month_summary

## 8. Top production companies

In [ ]:
company_summary = movies.groupby("company").agg(
    movie_count=("name", "count"),
    average_score=("score", "mean"),
    average_gross=("gross", "mean")
).sort_values("movie_count", ascending=False)

company_summary.head(20)

## 9. Top directors with enough movies

In [ ]:
director_summary = movies.groupby("director").agg(
    movie_count=("name", "count"),
    average_score=("score", "mean"),
    average_runtime=("runtime", "mean")
)

director_summary_reliable = director_summary[
    director_summary["movie_count"] >= 10
].sort_values("average_score", ascending=False)

director_summary_reliable.head(20)

## Industry analysis checklist

Use the tables above to write a short portfolio-ready summary:
1. What dominates the dataset by genre?
2. What does the financial distribution look like?
3. Is budget associated with gross?
4. Are high scores associated with revenue?
5. Which release months have the most observations?
6. Which companies/directors have enough observations for meaningful comparison?

Do not call a small-sample group the "best" or "worst"; describe the measured values and sample sizes.